# [BLOCK-T428] - LSSTCam IN-field stray light within a filter

This BLOCK assumes that the telescope is already pointed at a target and that it is already tracking.
It assumes that the optical systems are aligned to provide best focus.
The star must be in the center of the field. 

After that, repeat the following two steps for 53 times:
1. Offset telescope pointing by {el_offset} = -120 arcsec in elevation
2. Take an image with an exposure time of {exp_time}
   
At the end of the sequence, the bright star shall lay -1.76° off-axis.

Once the loop above is done, run the following two steps only once:
1. Bring back the bright star to the center of the FoV by re-pointing the telescope of {el_offset} = +6360 arcsec (+1.76°)
2. Take an image to check the star position.

[BLOCK-T428]: https://rubinobs.atlassian.net/projects/BLOCK?selectedItem=com.atlassian.plugins.atlassian-connect-plugin:com.kanoah.test-manager__main-project-page#!/v2/testCase/BLOCK-T428/testScript

In [ ]:
%load_ext autoreload
%autoreload 2
import warnings
import numpy as np
import os

from lsst.ts.block.utils import build_configuration_schema
from lsst.ts.observing import ObservingBlock, ObservingScript

In [ ]:
name = "BLOCK-T428"
program = "BLOCK-T428"
reason = "BLOCK-T428"
constraints = []
scripts = []

try:
    output_folder = (
        os.environ["TS_CONFIG_OCS_DIR"] + "/Scheduler/observing_blocks_maintel"
    )
except KeyError:
    warnings.warn(
        "The environment variable 'TS_CONFIG_OCS_DIR' is not set. Using default folder 'output_blocks'."
    )
    output_folder = "output_blocks"

In [ ]:
properties = {
    "el_offset": {
        "description": "Offset size that will be applied on every iteration.",
        "type": "number",
        "default": -120
    },
    "exp_time": {
        "description": "Exposure time used when taking images",
        "type": "number",
        "default": 15
    },
    "total_offset": {
        "description": "The total offset applied to bring the star back to the center of the field. `total_offset` = -53 * el_offset",
        "type": "number",
        "default": -53 * -120,
    }
}

block_number = name.split("-")[-1]
configuration_schema = build_configuration_schema(block_number, properties)

print(configuration_schema)

In [ ]:
offset_mttcs = ObservingScript(
    name="maintel/offset_mtcs.py",
    standard=True,
    parameters=dict(
        offset_azel=dict(
            az=0,
            el="$el_offset",    
        ), 
        relative=True,
    ),
)

take_image = ObservingScript(
    name="maintel/take_image_lsstcam.py",
    standard=True,
    parameters=dict(
        nimages=1,
        exp_times="$exp_time",
        image_type="ACQ",
        reason=reason,
        program=program
    )
)

# Apply several offsets and take images
scripts = []
for i in range(53):
    scripts.append(offset_mttcs)
    scripts.append(take_image)


# Bring back start to the center of the field
offset_mtts_back = ObservingScript(
    name="maintel/offset_mtcs.py",
    standard=True,
    parameters=dict(
        offset=dict(
           az=0,
            el="$total_offset", 
        ),
        relative=True,
    ),
)

scripts.append(offset_mtts_back)
scripts.append(take_image)

In [ ]:
block = ObservingBlock(
    name=name,
    program=program,
    scripts=scripts,
    constraints=constraints,
    configuration_schema=configuration_schema,
)

In [ ]:
block.model_dump_json(indent=2)

os.makedirs(output_folder, exist_ok=True)
output_path = f"{output_folder}/{name}.json"

with open(output_path, "w") as file:
    file.write(block.model_dump_json(indent=2))